# L1 (成分特征, v1) — 按化学式分组的 5 折交叉验证

与官方折协议相比, **只改了数据怎么分**:
- 外层: `GroupKFold(5)`, 按 reduced formula 分组 (划分来自 `make_group_splits.py`, L1/L2 共用)
- 内层早停验证集: `GroupShuffleSplit(test_size=0.2, random_state=42)`, 同样来自共用文件

其余不变: 特征 = run0709 的 v1 缓存 (不重新提取); 均值填补在**外层训练折**上计算 (与官方协议相同);
XGB 参数、SEED=42、上限 8000 轮、早停 200 轮、截断 均与主协议相同。
不调用 `task.record`, MAE/RMSE 自己算。SHAP 关闭。

In [1]:
# ==================== Cell 1: 导入 ====================
import json
import time
import hashlib
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import sklearn
from matbench.bench import MatbenchBenchmark
from matminer.featurizers.composition import ElementProperty
import xgboost as xgb

print('✅ 所有库导入成功!')

✅ 所有库导入成功!


~/miniforge3/envs/matbench3.9/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==================== Cell 2: 配置 ====================
PROTOCOL    = "group"
CONFIG_NAME = "L1"
TASK_NAME   = "matbench_mp_gap"
SEED        = 42
EXPECTED_N  = 106_113

NUM_BOOST_ROUND       = 8000     # 与主协议相同
EARLY_STOPPING_ROUNDS = 200

FEATURES_CACHE = Path("../outputs_v1_run0709") / "features_cache.pkl"   # run0709 的 v1 缓存 (只读)
SPLIT_DIR      = Path("../groupcv_splits")                              # make_group_splits.py 的输出
OUT_DIR        = Path("../outputs_v1_groupcv")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FOLDS_TO_RUN  = [0, 1, 2, 3, 4]
FORCE_RETRAIN = False            # 已有某折的预测和 json 就跳过训练, 直接读
N_VERIFY_PER_FOLD = 50           # 每个官方折抽多少条重算 magpie 特征, 核对缓存行没有错位

print(f'✅ 配置完成: {PROTOCOL} / {CONFIG_NAME} / seed {SEED} / 输出 {OUT_DIR}')

✅ 配置完成: group / L1 / seed 42 / 输出 ../outputs_v1_groupcv


In [3]:
# ==================== Cell 3: 加载数据和共用划分 ====================
mb   = MatbenchBenchmark(autoload=False)       # 只加载 mp_gap
task = next(t for t in mb.tasks if t.dataset_name == TASK_NAME)
task.load()
target_col = task.metadata["target"].strip()

df_all = task.df.rename(columns=lambda x: x.strip())
df_all.index = [str(i) for i in df_all.index]

folds = pd.read_csv(SPLIT_DIR / "folds.csv", dtype={"id": str, "formula": str}, float_precision="round_trip", keep_default_na=False)
inner = pd.read_csv(SPLIT_DIR / "inner_splits.csv", dtype={"id": str}, float_precision="round_trip", keep_default_na=False)
split_meta = json.loads((SPLIT_DIR / "split_meta.json").read_text(encoding="utf-8"))

# 划分文件必须和当前数据一一对应
assert len(folds) == EXPECTED_N and folds["id"].is_unique, "folds.csv 条数或 id 不对"
assert folds["id"].tolist() == df_all.index.tolist(), "folds.csv 的 id 顺序与 task.df 不一致"
assert np.array_equal(folds["y_true"].to_numpy(np.float64), df_all[target_col].to_numpy(np.float64)), "真值不一致"
_formula_now = [s.composition.reduced_formula for s in df_all["structure"]]
assert folds["formula"].tolist() == _formula_now, "folds.csv 的化学式与当前 pymatgen 计算结果不一致"

formula_by_id = folds.set_index("id")["formula"]
y_by_id       = folds.set_index("id")["y_true"]
print(f'✅ 数据 {len(df_all)} 条, 划分文件核对通过 (sklearn 生成时版本: {split_meta["versions"]["scikit-learn"]})')

2026-10-10 18:24:44 INFO     Initialized benchmark 'matbench_v0.1' with 13 tasks: 
['matbench_dielectric',
 'matbench_expt_gap',
 'matbench_expt_is_metal',
 'matbench_glass',
 'matbench_jdft2d',
 'matbench_log_gvrh',
 'matbench_log_kvrh',
 'matbench_mp_e_form',
 'matbench_mp_gap',
 'matbench_mp_is_metal',
 'matbench_perovskites',
 'matbench_phonons',
 'matbench_steels']
2026-10-10 18:24:44 INFO     Loading dataset 'matbench_mp_gap'...
2026-10-10 18:25:56 INFO     Dataset 'matbench_mp_gap loaded.
✅ 数据 106113 条, 划分文件核对通过 (sklearn 生成时版本: 1.6.1)


In [4]:
# ==================== Cell 4: 用缓存拼出全部 106,113 条的特征, 并核对没有错位 ====================
# v1 缓存按行号存 (没有 id), 行顺序 = 当时 get_test_data(fold) 的顺序。
# 官方 5 个测试折合起来正好每条出现一次, 按这个顺序贴上 id 再拼起来。
all_features = pd.read_pickle(FEATURES_CACHE)

parts = []
for fold in task.folds:
    tdf  = task.get_test_data(fold, as_type="df").rename(columns=lambda x: x.strip())
    tids = [str(i) for i in tdf.index]
    exp  = folds[folds["official_fold"] == int(fold)].sort_values("official_pos")["id"].tolist()
    assert tids == exp, f"官方 fold {fold} 的 id 顺序与 folds.csv 记录不一致"
    Xk = all_features[f"fold_{fold}_test"]
    if not isinstance(Xk, pd.DataFrame):
        Xk = pd.DataFrame(Xk)
    assert len(Xk) == len(tids), f"缓存 fold_{fold}_test 行数 {len(Xk)} ≠ {len(tids)}"
    Xk = Xk.copy()
    Xk.index = tids
    parts.append(Xk)

cols0 = list(parts[0].columns)
assert all(list(p.columns) == cols0 for p in parts), "各折缓存的特征列不一致"
X_all = pd.concat(parts, axis=0).loc[folds["id"]]          # 按数据集原始顺序
assert len(X_all) == EXPECTED_N and X_all.index.is_unique

# 缓存应是原始特征 (NaN 保留, 填补在训练折内做); 完全没有 NaN 说明可能是旧代码预先填补过
n_nan = int(X_all.isna().sum().sum())
print(f'  特征表: {X_all.shape}, NaN 个数 {n_nan}')
if n_nan == 0:
    print('  ⚠️ 缓存里没有任何 NaN, 可能是被旧代码预先填补过的版本, 请先确认再跑')

# --- 核对 1 (全量): magpie 最小/最大原子序数 必须等于这一行成分里的最小/最大原子序数 ---
comps = df_all.loc[X_all.index, "structure"].map(lambda s: s.composition)
zmin  = comps.map(lambda c: min(e.Z for e in c.elements)).to_numpy(np.float64)
zmax  = comps.map(lambda c: max(e.Z for e in c.elements)).to_numpy(np.float64)
assert np.array_equal(X_all["MagpieData minimum Number"].to_numpy(np.float64), zmin), "缓存行与结构错位 (min Z)"
assert np.array_equal(X_all["MagpieData maximum Number"].to_numpy(np.float64), zmax), "缓存行与结构错位 (max Z)"
print('  ✅ 全量核对通过: 每一行的 min/max 原子序数都和对应结构一致')

# --- 核对 2 (抽样): 重算整条 magpie 特征, 与缓存比较 ---
_ep  = ElementProperty.from_preset("magpie")
_lab = _ep.feature_labels()
_rng = np.random.RandomState(SEED)
n_ok = n_chk = 0
for fold in sorted(folds["official_fold"].unique()):
    ids_k = folds.loc[folds["official_fold"] == fold, "id"].to_numpy()
    for i in _rng.choice(ids_k, min(N_VERIFY_PER_FOLD, len(ids_k)), replace=False):
        new = np.asarray(_ep.featurize(df_all.at[i, "structure"].composition), dtype=np.float64)
        old = X_all.loc[i, _lab].to_numpy(np.float64)
        n_ok += bool(np.allclose(new, old, rtol=1e-6, atol=1e-9, equal_nan=True))
        n_chk += 1
print(f'  抽样重算: {n_ok}/{n_chk} 条完全一致')
if n_ok < 0.9 * n_chk:
    raise RuntimeError("抽样重算大量不一致, 缓存行可能错位, 停止")
elif n_ok < n_chk:
    print('  ⚠️ 少数不一致: 若差异很小, 多半是库版本差异而非错位; 请看具体差值后再决定')

FEATURE_COLUMNS = list(X_all.columns)
(OUT_DIR / "feature_columns.json").write_text(json.dumps(FEATURE_COLUMNS, ensure_ascii=False), encoding="utf-8")

  特征表: (106113, 132), NaN 个数 0
  ⚠️ 缓存里没有任何 NaN, 可能是被旧代码预先填补过的版本, 请先确认再跑
  ✅ 全量核对通过: 每一行的 min/max 原子序数都和对应结构一致


~/miniforge3/envs/matbench3.9/lib/python3.9/site-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


  抽样重算: 250/250 条完全一致


4168

In [5]:
# ==================== Cell 5: XGBoost 训练函数 ====================
# 与 v1 原函数相同, 只有一处改动: 内层 80/20 不再用 train_test_split 随机切,
# 而是用共用文件里的分组划分 (fit_mask)。另外返回训练信息供写 json。
def xgb_predict_with_early_stop(X_train, y_train, X_test, fold_idx: int, seed,
                                fit_mask, num_boost_round=8000, early_stopping_rounds=200,
                                verbose_eval=50):
    # [改动] 内层划分: 来自 inner_splits.csv (按化学式分组), 替代原来的
    #        train_test_split(X_train, y_train, test_size=0.2, random_state=seed, shuffle=True)
    X_tr, X_val = X_train[fit_mask], X_train[~fit_mask]
    y_tr, y_val = y_train[fit_mask], y_train[~fit_mask]

    dtr   = xgb.DMatrix(X_tr, label=y_tr)
    dval  = xgb.DMatrix(X_val, label=y_val)
    dtest = xgb.DMatrix(X_test)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "mae",
        "eta": 0.03,
        "max_depth": 6,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "alpha": 1e-3,
        "tree_method": "hist",
        "seed": seed,
    }

    evals_result = {}
    t0 = time.perf_counter()
    booster = xgb.train(
        params=params,
        dtrain=dtr,
        num_boost_round=num_boost_round,
        evals=[(dval, "validation")],
        evals_result=evals_result,
        early_stopping_rounds=early_stopping_rounds,
        verbose_eval=verbose_eval,
    )
    train_seconds = time.perf_counter() - t0

    best_it = booster.best_iteration
    best_score = booster.best_score
    print(f"[XGB Fold {fold_idx}] best_iteration = {best_it}, best_mae = {best_score:.6f}")

    try:
        y_pred = booster.predict(dtest, iteration_range=(0, best_it + 1))
    except TypeError:
        y_pred = booster.predict(dtest, ntree_limit=best_it + 1)

    val_mae_list = evals_result['validation']['mae']
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(val_mae_list) + 1), val_mae_list, label='Validation MAE', color='blue')
    plt.axvline(x=best_it + 1, color='red', linestyle='--', label=f'Best: {best_it} (MAE={best_score:.6f})')
    plt.xlabel('Boosting Rounds'); plt.ylabel('Validation MAE (eV)')
    plt.title(f'XGBoost Validation MAE Curve - {PROTOCOL} fold {fold_idx}')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.savefig(OUT_DIR / f"xgb_mae_curve_{PROTOCOL}_fold_{fold_idx}.png", dpi=150, bbox_inches='tight')
    plt.close()

    info = {
        "best_iteration": int(best_it),
        "best_validation_mae": float(best_score),
        "n_rounds_run": int(len(val_mae_list)),
        "num_boost_round_max": int(num_boost_round),
        "early_stopping_rounds": int(early_stopping_rounds),
        "early_stopped": bool(len(val_mae_list) < num_boost_round),
        "train_seconds": round(train_seconds, 3),
        "params": params,
    }
    return booster, y_pred, info

In [6]:
# ==================== Cell 6: 主流程 (5 折) ====================
predictions_dir = OUT_DIR / "predictions_xgb"
predictions_dir.mkdir(parents=True, exist_ok=True)
VERSIONS = {"python": platform.python_version(), "xgboost": xgb.__version__,
            "scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__}

def ids_sha256(ids):
    return hashlib.sha256("\n".join(map(str, ids)).encode("utf-8")).hexdigest()

fold_rows = {}
for k in FOLDS_TO_RUN:
    print(f'\n{"=" * 60}\n[{CONFIG_NAME}] group fold {k}\n{"=" * 60}')
    test_ids  = folds.loc[folds["group_fold"] == k, "id"].to_numpy()
    train_ids = folds.loc[folds["group_fold"] != k, "id"].to_numpy()
    inn = inner[inner["outer_fold"] == k].set_index("id")["role"]
    assert len(inn) == len(train_ids) and set(inn.index) == set(train_ids), "内层划分与外层训练折不对应"
    fit_mask = (inn.loc[train_ids] == "fit").to_numpy()

    # 三项检查 (每折都再查一遍)
    f_tr, f_te = set(formula_by_id.loc[train_ids]), set(formula_by_id.loc[test_ids])
    f_fit, f_val = set(formula_by_id.loc[train_ids[fit_mask]]), set(formula_by_id.loc[train_ids[~fit_mask]])
    assert not (f_tr & f_te), f"fold {k}: 外层训练/测试化学式有交集"
    assert not (f_fit & f_val), f"fold {k}: 内层 fit/val 化学式有交集"

    raw_file  = predictions_dir / f"pred_raw_fold_{k}.npy"
    clip_file = predictions_dir / f"pred_fold_{k}.npy"
    info_file = OUT_DIR / f"xgb_training_info_{PROTOCOL}_{CONFIG_NAME}_seed{SEED}_fold_{k}.json"
    y_test = y_by_id.loc[test_ids].to_numpy(np.float64)
    split_fp = {"test_ids_sha256": ids_sha256(test_ids), "fit_ids_sha256": ids_sha256(train_ids[fit_mask]),
                "val_ids_sha256": ids_sha256(train_ids[~fit_mask])}

    if raw_file.exists() and clip_file.exists() and info_file.exists() and not FORCE_RETRAIN:
        print('  已有结果, 跳过训练 (先核对它是不是用当前划分和设置跑出来的)')
        old = json.loads(info_file.read_text(encoding="utf-8"))
        for key, val in {**split_fp, "protocol": PROTOCOL, "config": CONFIG_NAME, "seed": SEED,
                         "num_boost_round_max": NUM_BOOST_ROUND,
                         "early_stopping_rounds": EARLY_STOPPING_ROUNDS}.items():
            if old.get(key) != val:
                raise RuntimeError(f"fold {k}: 已有结果的 {key} 与当前不一致 ({old.get(key)} vs {val}); "
                                   f"请换输出目录或设 FORCE_RETRAIN = True")
        y_raw  = np.load(raw_file)
        y_clip = np.load(clip_file)
    else:
        X_train = X_all.loc[train_ids]
        X_test  = X_all.loc[test_ids]
        y_train = y_by_id.loc[train_ids].to_numpy()

        # 填充: 统计量只从外层训练折来 (fit + val, 与官方折协议相同)
        col_means = X_train.mean(numeric_only=True)
        X_train = X_train.fillna(col_means).fillna(0)
        X_test  = X_test.fillna(col_means).fillna(0)

        booster, y_raw, tinfo = xgb_predict_with_early_stop(
            X_train, y_train, X_test, fold_idx=k, seed=SEED, fit_mask=fit_mask,
            num_boost_round=NUM_BOOST_ROUND, early_stopping_rounds=EARLY_STOPPING_ROUNDS)

        np.save(raw_file, y_raw)                 # clip 前的原始预测
        y_clip = np.maximum(y_raw, 0)            # 物理约束: 带隙非负
        np.save(clip_file, y_clip)

        err = y_clip.astype(np.float64) - y_test
        info = {
            "protocol": PROTOCOL, "config": CONFIG_NAME, "seed": SEED, "fold": int(k),
            **{key: tinfo[key] for key in ["best_iteration", "best_validation_mae", "n_rounds_run",
                                           "num_boost_round_max", "early_stopping_rounds",
                                           "early_stopped", "train_seconds"]},
            "MAE": float(np.mean(np.abs(err))), "RMSE": float(np.sqrt(np.mean(err ** 2))),
            "n_raw_negative": int((y_raw < 0).sum()),
            "n_train": int(len(train_ids)), "n_test": int(len(test_ids)),
            "n_fit": int(fit_mask.sum()), "n_val": int((~fit_mask).sum()),
            "val_fraction_of_outer_train": round(float((~fit_mask).mean()), 6),
            "n_groups_train": len(f_tr), "n_groups_test": len(f_te),
            "n_groups_fit": len(f_fit), "n_groups_val": len(f_val),
            "formula_overlap_train_test": 0, "formula_overlap_fit_val": 0,
            **split_fp,
            "impute_scope": "outer training fold (fit + val), same as official-fold protocol",
            "n_features": int(X_train.shape[1]),
            "params": tinfo["params"], "versions": VERSIONS,
        }
        info_file.write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f'  → MAE={info["MAE"]:.6f}  RMSE={info["RMSE"]:.6f}  best_it={info["best_iteration"]}  '
              f'early_stopped={info["early_stopped"]}')

    assert len(y_raw) == len(test_ids) and np.array_equal(np.maximum(y_raw, 0), y_clip)
    fold_rows[k] = pd.DataFrame({
        "protocol": PROTOCOL, "config": CONFIG_NAME, "seed": SEED, "fold": k, "id": test_ids,
        "formula": formula_by_id.loc[test_ids].to_numpy(),
        "formula_count_global": folds.set_index("id").loc[test_ids, "formula_count_global"].to_numpy(),
        "y_true": y_test, "y_raw": y_raw.astype(np.float64), "y_clip": y_clip.astype(np.float64),
    })


[L1] group fold 0
[0]	validation-mae:1.32062
[50]	validation-mae:0.77120
[100]	validation-mae:0.62489
[150]	validation-mae:0.58262
[200]	validation-mae:0.55985
[250]	validation-mae:0.54726
[300]	validation-mae:0.53581
[350]	validation-mae:0.52877
[400]	validation-mae:0.52205
[450]	validation-mae:0.51514
[500]	validation-mae:0.50873
[550]	validation-mae:0.50468
[600]	validation-mae:0.50038
[650]	validation-mae:0.49695
[700]	validation-mae:0.49460
[750]	validation-mae:0.49120
[800]	validation-mae:0.48818
[850]	validation-mae:0.48580
[900]	validation-mae:0.48342
[950]	validation-mae:0.48112
[1000]	validation-mae:0.47923
[1050]	validation-mae:0.47754
[1100]	validation-mae:0.47584
[1150]	validation-mae:0.47380
[1200]	validation-mae:0.47215
[1250]	validation-mae:0.47044
[1300]	validation-mae:0.46916
[1350]	validation-mae:0.46803
[1400]	validation-mae:0.46680
[1450]	validation-mae:0.46596
[1500]	validation-mae:0.46445
[1550]	validation-mae:0.46372
[1600]	validation-mae:0.46273
[1650]	validat

In [7]:
# ==================== Cell 7: 汇总 ====================
res = pd.concat([fold_rows[k] for k in sorted(fold_rows)], ignore_index=True)
per_fold = res.assign(ae=(res.y_clip - res.y_true).abs(), se=(res.y_clip - res.y_true) ** 2) \
              .groupby("fold").agg(n=("id", "size"), MAE=("ae", "mean"), RMSE=("se", "mean"))
per_fold["RMSE"] = np.sqrt(per_fold["RMSE"])
with pd.option_context("display.float_format", "{:.6f}".format):
    print(per_fold)
    print(f'\n五折平均 MAE = {per_fold.MAE.mean():.6f}, RMSE = {per_fold.RMSE.mean():.6f}')

if sorted(fold_rows) == [0, 1, 2, 3, 4]:
    assert len(res) == EXPECTED_N and res["id"].is_unique, "汇总后条数不是 106,113 或有重复 id"
    out_csv = OUT_DIR / f"pred_rows_{PROTOCOL}_{CONFIG_NAME}_seed{SEED}.csv"
    res.to_csv(out_csv, index=False)
    print(f'\n✅ 逐条预测已保存: {out_csv}  ({len(res)} 行)')
else:
    print(f'\n⚠️ 只跑了 fold {sorted(fold_rows)}, 不写逐条预测汇总文件')

          n      MAE     RMSE
fold                         
0     21223 0.383143 0.653697
1     21223 0.401309 0.675929
2     21223 0.403287 0.664903
3     21222 0.401929 0.660274
4     21222 0.396216 0.672729

五折平均 MAE = 0.397177, RMSE = 0.665507

✅ 逐条预测已保存: ../outputs_v1_groupcv/pred_rows_group_L1_seed42.csv  (106113 行)
